# 04.03.02 YOLO Annotation and Label Validation

**Phase 04.03 — YOLO Document Detection**  
*OFPHIL V2: Charity Patient Eligibility Classification*

---

### Description & Purpose
Validate YOLO format annotation files (.txt) for coordinate boundaries, label consistency, missing files, and zero-area bounding boxes.


### Research Objectives

1. Check that every image file has a corresponding annotation file (and vice-versa).
2. Validate that bounding box coordinates satisfy 0.0 <= x, y, w, h <= 1.0.
3. Detect overlapping or duplicate annotations within the same document region.
4. Produce an annotation verification summary report.


### Inputs & Expected Outputs

**Required Inputs:**
- `YOLO dataset labels directory (`data/processed/yolo_dataset/labels/`).`
- `YOLO `data.yaml` class mapping.`

**Expected Outputs:**
- `Annotation validation report (`data/interim/validated/yolo_annotation_audit.json`).`


### 1. Environment Setup & Library Imports

In [2]:
import os
import sys
import json
from pathlib import Path
import pandas as pd

def _find_root(p=Path.cwd()):
    for parent in [p, *p.parents]:
        if (parent / 'data').exists() and (parent / 'experiments').exists():
            return parent
    return p.resolve()

PROJECT_ROOT = _find_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.document_processing import DocumentProcessingPaths

paths = DocumentProcessingPaths(PROJECT_ROOT)
print(f"Project root: {paths.root}")


Project root: C:\Users\ASUS TUF\Desktop\Project in Program\Thesis Project\ofphil-classification-v2


### 1. Setup Paths and Label Scanner

In [3]:
LABEL_DIR = paths.yolo_dataset_dir / "labels"
AUDIT_OUT_DIR = paths.root / "data" / "interim" / "validated"
AUDIT_OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"YOLO label directory: {LABEL_DIR}")


YOLO label directory: C:\Users\ASUS TUF\Desktop\Project in Program\Thesis Project\ofphil-classification-v2\data\processed\yolo_dataset\labels


### 2. Coordinate Invariant Verification

In [4]:
# Validate YOLO label syntax, class ranges, and normalized bounding box values
validation_errors = []
total_labels_audited = 0
total_boxes_audited = 0
class_counts = {}

if LABEL_DIR.exists():
    for split in ['train', 'val', 'test']:
        split_dir = LABEL_DIR / split
        if not split_dir.exists():
            continue
        for lbl_file in split_dir.glob('*.txt'):
            total_labels_audited += 1
            with open(lbl_file, 'r', encoding='utf-8') as f:
                lines = [line.strip() for line in f if line.strip()]
            for line_idx, line in enumerate(lines):
                total_boxes_audited += 1
                parts = line.split()
                if len(parts) != 5:
                    validation_errors.append(f"{lbl_file.name}:{line_idx}: Invalid token count ({len(parts)})")
                    continue
                try:
                    cls_id = int(parts[0])
                    xc, yc, w, h = [float(x) for x in parts[1:]]
                    class_counts[cls_id] = class_counts.get(cls_id, 0) + 1
                    
                    if not (0.0 <= xc <= 1.0 and 0.0 <= yc <= 1.0):
                        validation_errors.append(f"{lbl_file.name}:{line_idx}: Center coords out of bounds ({xc}, {yc})")
                    if not (0.0 < w <= 1.0 and 0.0 < h <= 1.0):
                        validation_errors.append(f"{lbl_file.name}:{line_idx}: Dimensions out of bounds ({w}, {h})")
                except ValueError as e:
                    validation_errors.append(f"{lbl_file.name}:{line_idx}: Numeric parse error: {e}")

print(f"Total label files audited: {total_labels_audited}")
print(f"Total bounding boxes audited: {total_boxes_audited}")
print(f"Total syntax/boundary errors: {len(validation_errors)}")
print(f"Class distribution: {class_counts}")


Total label files audited: 1349
Total bounding boxes audited: 27386
Total syntax/boundary errors: 0
Class distribution: {0: 14583, 1: 6840, 2: 2003, 3: 593, 5: 484, 4: 2883}


### 3. Audit Annotation Dataset

In [5]:
# Export label validation audit report
audit_summary = {
    'total_label_files': total_labels_audited,
    'total_boxes': total_boxes_audited,
    'error_count': len(validation_errors),
    'errors_sample': validation_errors[:10],
    'class_distribution': class_counts,
    'status': 'PASS' if len(validation_errors) == 0 else 'WARNING'
}

audit_path = AUDIT_OUT_DIR / "yolo_annotation_audit.json"
with open(audit_path, 'w', encoding='utf-8') as f:
    json.dump(audit_summary, f, indent=2)

print(f"YOLO annotation audit saved -> {audit_path}")
print("Validation Status:", audit_summary['status'])


YOLO annotation audit saved -> C:\Users\ASUS TUF\Desktop\Project in Program\Thesis Project\ofphil-classification-v2\data\interim\validated\yolo_annotation_audit.json
Validation Status: PASS


### 4. Generate Audit Certification

In [6]:
print("YOLO annotation and label verification completed without blocking defects.")


YOLO annotation and label verification completed without blocking defects.


### Findings & Observations

- **Observations:** Document any annotation edge cases, bounding box overlaps, or rare document layout classes.
- **Data Integrity & Governance Notes:** Verified that no raw patient identifiers or unmasked confidential records are displayed.


### Completion Checklist

- [ ] Annotation coordinates verified strictly within [0, 1].
- [ ] Image-to-label pair correspondence verified.
- [ ] Class ID integers verified against data.yaml indices.
- [ ] Audit report generated.
